1) snapme + diabetes food hub + gpt api -> make alternative diets
2) train/test split

In [ ]:
import pandas as pd

mm = pd.read_csv("src/data/data3_multimodal/snapme_origin.csv")

In [215]:
mm.head(2)

,subject_id,before_caption,after_caption,recommendation,recommended_diet,before_image_path,after_image_path,general_info,food_identification_info,nutritional_info,food_classification_info,context_info,recommended_meal,instruction,input
0,SNAPME9001,coffee,coffee,### Analysis:\n\n**Strengths:**\n- Good source...,NaN,/data/jaesung/diabetes_rec/diellama/snapme/sna...,/data/jaesung/diabetes_rec/diellama/snapme/sna...,"{'snapme_study_day': 1, 'packaged_food': 0.0, ...","{'snapme_study_day': 1, 'packaged_food': 0.0, ...","{'PROT': 9.9324, 'TFAT': 6.0264, 'CARB': 27.60...","{'F_TOTAL': 0.0, 'F_JUICE': 0.0, 'V_STARCHY_TO...","Title: Citrus-Baked Pork Chops, Description: D...",Bountiful Harvest Vegetable Salad,Could you analyze the nutritional value of thi...,The image before the meal can be found at /dat...
1,SNAPME9001,orange,peeled orange,### Analysis:\n- **Strengths**:\n - Good sour...,- **Strengths**:,/data/jaesung/diabetes_rec/diellama/snapme/sna...,/data/jaesung/diabetes_rec/diellama/snapme/sna...,"{'snapme_study_day': 1, 'packaged_food': 0.0, ...","{'snapme_study_day': 1, 'packaged_food': 0.0, ...","{'PROT': 1.2314, 'TFAT': 0.1572, 'CARB': 15.39...","{'F_TOTAL': 0.7074, 'F_JUICE': 0.0, 'V_STARCHY...","Title: Just Peachy Yogurt and Granola Jar, Des...",Grilled Lime Chicken Fajitas,Could you provide an analysis of this meal and...,The pre-meal image is located at /data/jaesung...


In [ ]:
import pandas as pd
from ast import literal_eval
import numpy as np

def extract_food_description(row):
    try:
        row = row.replace("nan", "None")
        data = literal_eval(row)
        return data.get('Food_Description', None) 
    except (ValueError, SyntaxError, TypeError) as e:
        print(f"Error parsing row: {row}, Error: {e}")
        return None 

mm['food_identification_info'] = mm['food_identification_info'].astype(str) 

mm2 = pd.DataFrame({
    'input': mm['food_identification_info'].apply(extract_food_description),
    'input_nutrition_facts': mm['nutritional_info'],
    'output': mm['recommended_meal']
})

In [217]:
mm2.head()

,input,input_nutrition_facts,output
0,"Coffee, Latte, flavored","{'PROT': 9.9324, 'TFAT': 6.0264, 'CARB': 27.60...",Bountiful Harvest Vegetable Salad
1,"Orange, raw","{'PROT': 1.2314, 'TFAT': 0.1572, 'CARB': 15.39...",Grilled Lime Chicken Fajitas
2,"Cheese, cottage, low fat","{'PROT': 17.71275, 'TFAT': 3.84765, 'CARB': 8....",Avocado Toast with Turkey Bacon and Tomato
3,"Blueberries, raw","{'PROT': 0.2738, 'TFAT': 0.1221, 'CARB': 5.361...",Air Fryer Spicy Green Beans
4,"Lettuce, arugula, raw","{'PROT': 0.3612, 'TFAT': 0.0924, 'CARB': 0.511...",Grilled Lime Chicken Fajitas


In [ ]:
import pandas as pd
import ast
import re

NUTRITION_UNITS = {
    'PROT': 'g',   # 단백질
    'TFAT': 'g',   # 총 지방
    'CARB': 'g',   # 탄수화물
    'SUGR': 'g',   # 당
    'FIBE': 'g',   # 식이섬유
    'MAGN': 'mg',  # 마그네슘
    'POTA': 'mg',  # 칼륨
    'VB1': 'mg',   # 비타민 B1
    'VB6': 'mg',   # 비타민 B6
    'VB12': 'µg',  # 비타민 B12
    'SFAT': 'g'    # 포화 지방
}

def extract_context_info(context, title):
    try:
        entries = context.split("\n")
        for entry in entries:
            if f"Title: {title}," in entry:
                description = entry.split("Description:")[1].split(", Nutrition Facts:")[0].strip()
                nutrition_facts_str = entry.split("Nutrition Facts:")[1].split(", Ingredients:")[0].strip()
                nutrition_facts = ast.literal_eval(nutrition_facts_str)  # Nutrition Facts를 딕셔너리로 변환
                return description, nutrition_facts
    except Exception as e:
        print(f"Error processing title: {title}, Error: {e}")
    return None, None  

def scale_to_100g_with_units(nutrition_facts, amount_per_serving):
    """Nutrition Facts 값을 100g 기준으로 변환 (단위를 유지)"""
    if not isinstance(nutrition_facts, dict) or not isinstance(amount_per_serving, (int, float)):
        return nutrition_facts
    scaled_facts = {}
    for key, value in nutrition_facts.items():
        try:
            numeric_value = float(re.findall(r"[-+]?\d*\.\d+|\d+", value)[0])
            unit = re.sub(r"[-+]?\d*\.\d+|\d+", "", value).strip()
            # 100g 기준으로 변환
            scaled_value = (numeric_value / amount_per_serving) * 100
            scaled_facts[key] = f"{scaled_value:.2f} {unit}" if unit else scaled_value
        except (ValueError, IndexError):
            scaled_facts[key] = value
    return scaled_facts

def add_units_to_nutrition_facts(nutrition_facts):
    if not isinstance(nutrition_facts, dict):
        return nutrition_facts  
    with_units = {}
    for key, value in nutrition_facts.items():
        unit = NUTRITION_UNITS.get(key, '')  
        with_units[key] = f"{value} {unit}" if unit else value
    return with_units

output_descriptions = []
output_nutrition_facts_scaled = []

for i, row in mm.iterrows():
    title = row['recommended_meal']
    context = row['context_info']
    description, nutrition_facts = extract_context_info(context, title)
    output_descriptions.append(description)
    if nutrition_facts:
        try:
            amount_per_serving = float(nutrition_facts.get('Amount per serving', '100').strip('g'))  # 기본값 100g
            # 100g 기준으로 변환
            scaled_nutrition = scale_to_100g_with_units(nutrition_facts, amount_per_serving)
            output_nutrition_facts_scaled.append(scaled_nutrition)
        except Exception as e:
            print(f"Error scaling nutrition facts for {title}, Error: {e}")
            output_nutrition_facts_scaled.append(None)
    else:
        output_nutrition_facts_scaled.append(None)

mm2['input_nutrition_facts'] = mm2['input_nutrition_facts'].apply(lambda x: add_units_to_nutrition_facts(ast.literal_eval(x)))

mm2['output_description'] = output_descriptions
mm2['output_nutrition_facts'] = output_nutrition_facts_scaled


In [219]:
mm2.head(2)

,input,input_nutrition_facts,output,output_description,output_nutrition_facts
0,"Coffee, Latte, flavored","{'PROT': '9.9324 g', 'TFAT': '6.0264 g', 'CARB...",Bountiful Harvest Vegetable Salad,This is a wonderful way to use fresh produce f...,"{'Amount per serving': 100.0, 'Total Fat': '6...."
1,"Orange, raw","{'PROT': '1.2314 g', 'TFAT': '0.1572 g', 'CARB...",Grilled Lime Chicken Fajitas,Chicken fajitas are one of the best choices wh...,"{'Amount per serving': 100.0, 'Total Fat': '2...."


In [ ]:
import openai
import pandas as pd
import json
from tqdm import tqdm

from dotenv import load_dotenv
import os

load_dotenv()

openai.api_key = os.getenv("OPENAI_API_KEY")

def generate_reason(input_food, input_facts, output_food, output_desc, output_facts):
    try:
        prompt = (
            f"The consumed food '{input_food}' has certain nutritional deficiencies as detailed in: {input_facts}. "
            f"The recommended food '{output_food}' can help address these deficiencies based on the following details: "
            f"Description: {output_desc}, Nutrition Information: {output_facts}. "
            f"Explain briefly why '{output_food}' complements '{input_food}' without mentioning exact nutritional values. "
            f"If there are any notable drawbacks of '{output_food}', mention them as cautionary points. "
            f"Write a factual and concise explanation in 1-2 sentences."
        )
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo-0125",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=70,
            temperature=0.7
        )
        return response['choices'][0]['message']['content'].strip()
    except Exception as e:
        return f"Error: {e}"

def save_to_json(file_path, data):
    try:
        try:
            with open(file_path, "r") as f:
                existing_data = json.load(f)
        except FileNotFoundError:
            existing_data = []  

        existing_data.append(data)

        with open(file_path, "w") as f:
            json.dump(existing_data, f, indent=4)
    except Exception as e:
        print(f"Error saving to JSON: {e}")

mm2 = mm2

file_path = "snapme_origin_v1.json"

def process_and_save():
    for idx, row in tqdm(mm2.iterrows(), total=mm2.shape[0], desc="Processing rows"):
        reason = generate_reason(
            row['input'], row['input_nutrition_facts'],
            row['output'], row['output_description'],
            row['output_nutrition_facts']
        )

        result = {
            "input": row['input'],
            "input_nutrition_facts": row['input_nutrition_facts'],
            "output": row['output'],
            "output_description": row['output_description'],
            "output_nutrition_facts": row['output_nutrition_facts'],
            "reason": reason
        }

        save_to_json(file_path, result)

process_and_save()

print(f"Reason generation complete. Results saved to {file_path}.")


Processing rows:   0%|          | 0/4317 [00:00<?, ?it/s]

Processing rows: 100%|██████████| 4317/4317 [1:24:06<00:00,  1.17s/it]

Reason generation complete. Results saved to snapme_origin_v1.json.


In [223]:
file_path = "/data/jaesung/llm_for_diabetes/src/data/data3_multimodal/snapme_origin_v1.json"
with open(file_path, 'r', encoding='utf-8') as f:
    tmp = json.load(f)

mm3 = pd.DataFrame(tmp)

In [224]:
mm3.head()

,input,input_nutrition_facts,output,output_description,output_nutrition_facts,reason
0,"Coffee, Latte, flavored","{'PROT': '9.9324 g', 'TFAT': '6.0264 g', 'CARB...",Bountiful Harvest Vegetable Salad,This is a wonderful way to use fresh produce f...,"{'Amount per serving': 100.0, 'Total Fat': '6....",The Bountiful Harvest Vegetable Salad compleme...
1,"Orange, raw","{'PROT': '1.2314 g', 'TFAT': '0.1572 g', 'CARB...",Grilled Lime Chicken Fajitas,Chicken fajitas are one of the best choices wh...,"{'Amount per serving': 100.0, 'Total Fat': '2....",'Grilled Lime Chicken Fajitas' complements 'Or...
2,"Cheese, cottage, low fat","{'PROT': '17.71275 g', 'TFAT': '3.84765 g', 'C...",Avocado Toast with Turkey Bacon and Tomato,Avocado toast is a quick and easy go-to breakf...,"{'Amount per serving': 100.0, 'Total Fat': '56...",'Avocado Toast with Turkey Bacon and Tomato' c...
3,"Blueberries, raw","{'PROT': '0.2738 g', 'TFAT': '0.1221 g', 'CARB...",Air Fryer Spicy Green Beans,Spice up your green beans with this air fried ...,"{'Amount per serving': 100.0, 'Total Fat': '5....",'Air Fryer Spicy Green Beans' complements 'Blu...
4,"Lettuce, arugula, raw","{'PROT': '0.3612 g', 'TFAT': '0.0924 g', 'CARB...",Grilled Lime Chicken Fajitas,Chicken fajitas are one of the best choices wh...,"{'Amount per serving': 100.0, 'Total Fat': '2....",'Grilled Lime Chicken Fajitas' complements 'Le...


In [225]:
len(mm3)

4317

In [ ]:
from sklearn.model_selection import train_test_split

mm3_train, mm3_test = train_test_split(mm3, test_size=0.2, random_state=42)

In [231]:
mm3_train.head(2)

,input,input_nutrition_facts,output,output_description,output_nutrition_facts,reason
279,"Cheese, Parmesan, dry grated","{'PROT': '8.05707 g', 'TFAT': '7.89264 g', 'CA...",Grilled Lime Chicken Fajitas,Chicken fajitas are one of the best choices wh...,"{'Amount per serving': 100.0, 'Total Fat': '2....",'Grilled Lime Chicken Fajitas' complements 'Ch...
3435,"Tomatoes, raw","{'PROT': '0.48664 g', 'TFAT': '0.1106 g', 'CAR...",Bountiful Harvest Vegetable Salad,This is a wonderful way to use fresh produce f...,"{'Amount per serving': 100.0, 'Total Fat': '6....",'Bountiful Harvest Vegetable Salad' complement...


In [ ]:
import pandas as pd
import json

def generate_json_from_df(df):
    results = []
    for _, row in df.iterrows():
        result = {
            "instruction": "Based on the previous meal, suggest the next meal to maintain a balanced diet.",
            "input": row["input"],
            "output": f"{row['output']} is recommended. The reason is {row['reason']}"
        }
        results.append(result)
    return results

json_results = generate_json_from_df(mm3_train)

with open("snapme_train_instruction_dataset.json", "w", encoding="utf-8") as f:
    json.dump(json_results, f, ensure_ascii=False, indent=4)

print(json.dumps(json_results[:2], ensure_ascii=False, indent=4))


[
    {
        "instruction": "Based on the previous meal, suggest the next meal to maintain a balanced diet.",
        "input": "Cheese, Parmesan, dry grated",
        "output": "Grilled Lime Chicken Fajitas is recommended. The reason is 'Grilled Lime Chicken Fajitas' complements 'Cheese, Parmesan, dry grated' as it provides a lean protein source, low-carb vegetables, and a healthier cooking method. However, caution should be taken with the sodium content in the fajitas, as excessive sodium intake can lead to health issues."
    },
    {
        "instruction": "Based on the previous meal, suggest the next meal to maintain a balanced diet.",
        "input": "Tomatoes, raw",
        "output": "Bountiful Harvest Vegetable Salad is recommended. The reason is 'Bountiful Harvest Vegetable Salad' complements 'Tomatoes, raw' because it provides a balanced combination of protein, healthy fats, and carbohydrates that can help address the deficiencies in the raw tomatoes. However, individuals 

In [ ]:
import pandas as pd
import json

def generate_json_from_df(df):
    results = []
    for _, row in df.iterrows():
        result = {
            "instruction": "Based on the previous meal, suggest the next meal to maintain a balanced diet.",
            "input": row["input"],
            "output": f"{row['output']} is recommended. The reason is {row['reason']}"
        }
        results.append(result)
    return results

json_results = generate_json_from_df(mm3_test)

with open("snapme_test_instruction_dataset.json", "w", encoding="utf-8") as f:
    json.dump(json_results, f, ensure_ascii=False, indent=4)

print(json.dumps(json_results[:2], ensure_ascii=False, indent=4))


[
    {
        "instruction": "Based on the previous meal, suggest the next meal to maintain a balanced diet.",
        "input": "Flax seeds",
        "output": "Orange, Asparagus, and Avocado Salad is recommended. The reason is 'Orange, Asparagus, and Avocado Salad' complements 'Flax seeds' as it provides a good source of healthy fats, fiber, and vitamins, which can help address the deficiencies in the consumed food. However, caution should be taken with the fat content in the salad, as it may be higher than desired for some individuals."
    },
    {
        "instruction": "Based on the previous meal, suggest the next meal to maintain a balanced diet.",
        "input": "Lettuce, arugula, raw",
        "output": "Budget-Friendly Vegetable Stew with Whole Wheat Dumplings is recommended. The reason is The Budget-Friendly Vegetable Stew with Whole Wheat Dumplings complements Lettuce, arugula, raw by providing a good source of protein, fat, and carbohydrates that are lacking in the cons